# Fase 2: RAG

## 1. Chatbot básico

### 1.1. Configuración del cliente de Groq

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
API_KEY = os.getenv("GROQ_KEY")

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=API_KEY)

### 1.2. Comparar variantes de system prompt

In [3]:
SYSTEM_PROMPTS = {
    "Asistente General": (
        "Eres un asistente útil que responde preguntas de forma clara y comprensible. "
        "Proporciona respuestas correctas y breves. "
        "No es necesario usar un lenguaje técnico."
    ),
    "Asistente Especializado": (
        "Eres un asistente experto en privacidad digital, protección de datos y términos de servicio. "
        "Responde de forma breve, precisa, clara y estructurada. "
        "Utiliza lenguaje técnico cuando sea necesario, pero mantén la claridad. "
    ),
    "Asistente Restrictivo": (
        "Eres un asistente especializado en privacidad, protección de datos y términos de servicio."
        "Responde únicamente a preguntas relacionadas con estos temas."
        "Si la pregunta no está relacionada con privacidad, protección de datos o términos digitales, debes responder exactamente:"
        "\"No dispongo de información suficiente para responder con precisión.\""
        "No respondas preguntas fuera de este dominio bajo ninguna circunstancia."
        "Usa un tono formal y respuestas breves y estructuradas."
    ),
}

PREGUNTAS = [
    # Preguntas del dominio
    "¿Qué son las cookies y para qué se utilizan?",
    "¿Qué derechos tiene un usuario según el RGPD?",
    "¿Qué datos personales recopila Spotify y con qué finalidad?",

    # Preguntas fuera del dominio (para analizar alucinaciones y límites del modelo)
    "¿Cómo funciona la fotosíntesis en las plantas?",
    "¿Cómo se relacionan la teoría de la relatividad y la mecánica cuántica en la descripción del espacio-tiempo?"
]

In [6]:
MODEL = "llama-3.1-8b-instant"

def generar_respuesta(pregunta, system_prompt, temperature=1.0):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": pregunta}
        ],
        max_completion_tokens=300,
        temperature=temperature
    )
    return response.choices[0].message.content.strip()

In [37]:
# Comparar respuestas de los tres asistentes para cada pregunta
for pregunta in PREGUNTAS:
    print(f"\nPregunta: {pregunta}\n{'='*50}")
    for asistente, system_prompt in SYSTEM_PROMPTS.items():
        respuesta = generar_respuesta(pregunta, system_prompt)
        print(f"{asistente}:\n{respuesta}\n{'-'*50}")


Pregunta: ¿Cómo se relacionan la teoría de la relatividad y la mecánica cuántica en la descripción del espacio-tiempo?
Asistente General:
La teoría de la relatividad y la mecánica cuántica son dos marcos teóricos fundamentales de la física que se relacionan en la descripción del espacio-tiempo de manera importante.

La teoría de la relatividad general, desarrollada por Albert Einstein, describe el espacio-tiempo como una sola entidad curva y dinámica, influenciada por la presencia de masa y energía. Esto implica que el espacio y el tiempo no son absolutos, sino que están interconectados y se curvan en presencia de objetos masivos.

Por otro lado, la mecánica cuántica describe los comportamientos de los objetos a nivel atómico y subatómico, donde las partículas se comportan de manera probabilística y no determinista. La mecánica cuántica también tiene un impacto en la descripción del espacio-tiempo, ya que sugiere que el espacio-t
--------------------------------------------------
Asis

### 1.3. Comparar parámetros de generación

In [11]:
# Se selecciona una pregunta que permita evaluar precisión + control + alucinaciones
pregunta = PREGUNTAS[2]
system_prompt = SYSTEM_PROMPTS["Asistente Restrictivo"]

print("Temperatura = 1.0 (respuestas más creativas y variadas)\n")
print(generar_respuesta(pregunta, system_prompt, temperature=1.0))

print(f"\n{'='*50}\nTemperatura = 0.6 (respuestas más coherentes y menos creativas)\n")
print(generar_respuesta(pregunta, system_prompt, temperature=0.6))

print(f"\n{'='*50}\nTemperatura = 0.2 (respuestas más precisas y menos creativas)\n")
print(generar_respuesta(pregunta, system_prompt, temperature=0.2))

Temperatura = 1.0 (respuestas más creativas y variadas)

Spotify recopila diversos tipos de datos personales, incluyendo:

1. **Información de cuenta**:nombre de usuario, contraseña, correo electrónico, fecha de nacimiento y género.
2. **Datos de actividad**: reproducciones de canciones, listas de reproducción, búsqueda de contenidos y duración de la sesión.
3. **Información de dispositivo**: tipo de dispositivo, idioma, versión de sistema operativo y datos de localización.
4. **Datos de pago**: información de tarjeta de crédito o débito, método de pago y saldo.

El fin principal de recopilar estos datos es proporcionar servicios a los usuarios, como:

- Personalizar recomendaciones de música.
- Permitir el acceso a contenido premium.
- Mejorar la experiencia de usuario y el rendimiento del servicio.
- Cumplir con obligaciones legales y regulaciones, como las relacionadas con la protección de menores.

No obstante, Spotify ha publicado una política de privacidad detallada, disponible e

 **Conclusión:** La mejor combinación para RAG es "Asistente Restrictivo" como system prompt + temperature=0.2

## 2. Chunking + embeddings

En este apartado se compararán los métodos de chunking por tamaño fijo y recursivo mostrando como quedan los chunks y evaluando cualitativamente la calidad del retrieval. 

Para ello, se usará solo uno de los documentos del corpus. 

Una vez seleccionada la estrategia de chunking más óptima, se implementará en el apartado 3 el pipeline completo almacenando los embeddings en una base de datos vectorial.

### 2.1. Comparar configuraciones de chunking 

In [51]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Chunking por tamaño fijo
splitter_fijo = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,      # máximo de caracteres por chunk
    chunk_overlap=50,    # solapamiento entre chunks consecutivos
    length_function=len
)

# Chunking recursivo
splitter_recursivo = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # máximo de caracteres por chunk
    chunk_overlap=150,   # 15% de solapamiento
    # Prueba separadores de mayor a menor granularidad
    separators=["\n# ", "\n## ", "\n### ", "\n---\n", "\n\n", "\n", ". ", " "],
    length_function=len,
)

In [52]:
# Comparar chunks generados por cada método
def mostrar_chunks(chunks):
    print(f"Chunks generados: {len(chunks)}\n")
    # Mostrar chunks del 20 al 25 para ver variedad
    for i, chunk in enumerate(chunks[19:25]):
        print(f"--- Chunk {i + 20} ({len(chunk)} chars) ---")
        print(chunk)
        print()

# Para comparar ambos metodos, se usa uno de los documentos del corpus
with open("corpus/guides/gdpr_explained.md", "r", encoding="utf-8") as f:
    texto = f.read()

# Generar chunks usando ambos métodos
chunks_fijo = splitter_fijo.split_text(texto)
chunks_recursivo = splitter_recursivo.split_text(texto)


Created a chunk of size 760, which is longer than the specified 500


In [53]:
print("Chunks generados por el método de tamaño fijo:")
mostrar_chunks(chunks_fijo)

Chunks generados por el método de tamaño fijo:
Chunks generados: 34

--- Chunk 20 (331 chars) ---
### Data protection by design and by default
From now on, everything you do in your organization must, “by design and by default,” consider data protection. Practically speaking, this means you must consider the data protection principles in the design of any new product or activity. The GDPR covers this principle in Article 25 .

--- Chunk 21 (297 chars) ---
Suppose, for example, you’re launching a new app for your company. You have to think about what personal data the app could possibly collect from users, then consider ways to minimize the amount of data and how you will secure it with the latest technology.
### When you’re allowed to process data

--- Chunk 22 (424 chars) ---
### When you’re allowed to process data
Article 6 lists the instances in which it’s legal to process person data. Don’t even think about touching somebody’s personal data — don’t collect it, don’t store it, don’t

In [54]:
print("Chunks generados por el método recursivo:")
mostrar_chunks(chunks_recursivo)

Chunks generados por el método recursivo:
Chunks generados: 26

--- Chunk 20 (279 chars) ---
Once you’ve determined the lawful basis for your data processing, you need to document this basis and notify the data subject (transparency!). And if you decide later to change your justification, you need to have a good reason, document this reason, and notify the data subject.

--- Chunk 21 (651 chars) ---
### Consent

There are strict new rules about what constitutes consent from a data subject to process their information.

- Consent must be “freely given, specific, informed and unambiguous.”
- Requests for consent must be “clearly distinguishable from the other matters” and presented in “clear and plain language.”
- Data subjects can withdraw previously given consent whenever they want, and you have to honor their decision. You can’t simply change the legal basis of the processing to one of the other justifications.
- Children under 13 can only give consent with permission from their paren

In [47]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer("intfloat/multilingual-e5-base")

# Generar embeddings de chunks 
def generar_embeddings(chunks):
    list_chunks = ["passage: " + c for c in chunks]
    embeddings = modelo_emb.encode(
        list_chunks,
        normalize_embeddings=True
    )
    return embeddings

embeddings_chunks_fijo = generar_embeddings(chunks_fijo)
embeddings_chunks_recursivo = generar_embeddings(chunks_recursivo)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6620.50it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
# Evaluar cualitativamente la calidad del retrieval para cada método de chunking 
import numpy as np

QUERIES = [
    "What is the GDPR?",
    "Which are the data protection principles?",
    "What are the rights of a user according to the GDPR?",
]

def mostrar_similitudes(embeddings, chunks, queries, top_k=3):
    """Muestra los chunks más similares a cada query según los embeddings."""
    for query in queries:
        query_emb = modelo_emb.encode(["query: " + query], normalize_embeddings=True)[0]

        # Similitud coseno entre la query y todos los documentos
        similitudes = modelo_emb.similarity(query_emb, embeddings)[0].numpy()
        
        # Índices ordenados de mayor a menor similitud
        top_indices = np.argsort(similitudes)[-top_k:][::-1]

        print(f"\nQuery: {query}\n{'-'*50}")
        for i, idx in enumerate(top_indices):
            print(f"{i + 1}. (sim={similitudes[idx]:.3f}) {chunks[idx]}")


In [56]:
print("Similitudes para chunks generados por el método de tamaño fijo:")
mostrar_similitudes(embeddings_chunks_fijo, chunks_fijo, QUERIES)

Similitudes para chunks generados por el método de tamaño fijo:

Query: What is the GDPR?
--------------------------------------------------
1. (sim=0.907) The General Data Protection Regulation (GDPR) is the toughest privacy and security law in the world. Though it was drafted and passed by the European Union (EU), it imposes obligations onto organizations anywhere, so long as they target or collect data related to people in the EU. The regulation was put into effect on May 25, 2018. The GDPR will levy harsh fines against those who violate its privacy and security standards, with penalties reaching into the tens of millions of euros.
2. (sim=0.890) If you’ve found this page — “what is the GDPR?” — chances are you’re looking for a crash course. Maybe you haven’t even found the document itself yet (tip: here’s the full regulation ). Maybe you don’t have time to read the whole thing. This page is for you. In this article, we try to demystify the GDPR and, we hope, make it less overwhelmi

In [57]:
print("Similitudes para chunks generados por el método recursivo:")
mostrar_similitudes(embeddings_chunks_recursivo, chunks_recursivo, QUERIES)

Similitudes para chunks generados por el método recursivo:

Query: What is the GDPR?
--------------------------------------------------
1. (sim=0.907) The General Data Protection Regulation (GDPR) is the toughest privacy and security law in the world. Though it was drafted and passed by the European Union (EU), it imposes obligations onto organizations anywhere, so long as they target or collect data related to people in the EU. The regulation was put into effect on May 25, 2018. The GDPR will levy harsh fines against those who violate its privacy and security standards, with penalties reaching into the tens of millions of euros.
2. (sim=0.893) # What is GDPR, the EU’s new data protection law?

What is the GDPR? Europe’s new data privacy and security law includes hundreds of pages’ worth of new requirements for organizations around the world. This GDPR overview will help you understand the law and determine what parts of it apply to you.
3. (sim=0.881) With the GDPR, Europe is signalin

**Conclusiones:** El método recursivo genera chunks más coherentes y relevantes para las queries, lo que mejora la calidad del retrieval. El método de tamaño fijo produce chunks arbitrarios que pueden cortar información importante, resultando en similitudes más bajas y peor retrieval al devolver chunks menos relevantes.

Por ello, en el pipeline completo se aplicará chunking recursivo con 20% de solapamiento entre chunks.

## 3. Pipeline RAG completo

### 3.1. Chunking + Embeddings de todos los documentos del corpus

In [3]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunking_documento(doc_path):
    """
    Realiza chunking recursivo con solapamiento para un documento,
    devolviendo chunks junto con metadatos de archivo, categoria y empresa.
    """
    # Extraer nombre del archivo y categoría para metadatos
    ruta = Path(doc_path)
    archivo = ruta.name
    categoria = ruta.parent.name

    # Guardar empresa como metadato adicional
    nombre_archivo = archivo.lower()
    if "google" in nombre_archivo:
        empresa = "Google"
    elif "spotify" in nombre_archivo:
        empresa = "Spotify"
    else:
        empresa = None

    # Leer el contenido del documento
    with open(doc_path, "r", encoding="utf-8") as f:
        texto = f.read()

    # Chunking recursivo
    splitter_recursivo = RecursiveCharacterTextSplitter(
        chunk_size=1000,      # máximo de caracteres por chunk
        chunk_overlap=150,    # % de solapamiento
        # Prueba separadores de mayor a menor granularidad
        separators=["\n# ", "\n## ", "\n### ", "\n---", "\n\n", "\n", ". ", " "],
        length_function=len,
    )
    chunks = splitter_recursivo.split_text(texto)

    # En algunos casos, los titulos quedan aislados como chunks muy cortos. 
    # Se unen al siguiente chunk para mejorar contexto.
    chunks_unidos = []
    buffer = ""
    for chunk in chunks:
        # si es título (empieza por #) y chunk corto
        if chunk.startswith("#") and len(chunk) < 100:
            buffer = chunk
        else:
            if buffer:
                chunks_unidos.append(buffer + "\n" + chunk)
                buffer = ""
            else:
                chunks_unidos.append(chunk)
    if buffer:
        chunks_unidos.append(buffer)

    # Devolver chunks + metadatos
    return {
        "archivo": archivo,
        "categoria": categoria,
        "empresa": empresa,
        "chunks": chunks_unidos,
    }


def chunking_corpus(corpus_dir):
    """
    Realiza el proceso de chunking para todos los documentos .md en un directorio dado, 
    devolviendo una lista de diccionarios con los chunks y metadatos asociados a cada documento.
    """
    corpus = []
    for doc_path in Path(corpus_dir).rglob("*.md"):
        doc_chunks = chunking_documento(doc_path)
        corpus.append(doc_chunks)
    return corpus


def generar_embeddings_corpus(chunks_corpus, modelo_emb):
    """
    Genera embeddings para todos los chunks del corpus y preserva metadata.
    """
    flat_chunks = []

    # Aplanar manteniendo metadata
    for doc in chunks_corpus:
        for chunk in doc["chunks"]:
            flat_chunks.append({
                "chunk": chunk,
                "archivo": doc["archivo"],
                "categoria": doc["categoria"],
                "empresa": doc["empresa"]
            })

    # Preparar textos para E5
    chunks_e5 = ["passage: " + c["chunk"] for c in flat_chunks]

    # Embeddings en batch
    embeddings = modelo_emb.encode(
        chunks_e5,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    # Unir embeddings + metadata
    embeddings_metadata = []
    for i, item in enumerate(flat_chunks):
        embeddings_metadata.append({
            "embedding": embeddings[i],
            "chunk": item["chunk"],
            "archivo": item["archivo"],
            "categoria": item["categoria"],
            "empresa": item["empresa"]
        })

    return embeddings_metadata

c:\Users\andre\Documents\GitHub\PECL_LLMs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

modelo_emb = SentenceTransformer("intfloat/multilingual-e5-base")
sparse_model = SparseTextEmbedding("Qdrant/bm25")
corpus_dir = "corpus/"

# Chunking + embeddings para todo el corpus
chunks_corpus = chunking_corpus(corpus_dir)
embeddings_corpus = generar_embeddings_corpus(chunks_corpus, modelo_emb)

# Sparse vectors para busqueda BM25
textos = [item["chunk"] for item in embeddings_corpus]
sparse_vectors = list(sparse_model.embed(textos))

print(f"Documentos procesados: {len(chunks_corpus)}")
print(f"Chunks con embedding: {len(embeddings_corpus)}")
print(f"Vectores sparse generados: {len(sparse_vectors)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3875.57it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 9/9 [00:38<00:00,  4.23s/it]

Documentos procesados: 6
Chunks con embedding: 268
Vectores sparse generados: 268


### 3.2. Base de datos vectorial Qdrant

Levantar el servidor Qdrant con Docker

PowerShell (Windows):

```powershell
docker run -p 6333:6333 -p 6334:6334 -v ${PWD}/qdrant_storage:/qdrant/storage qdrant/qdrant
```

Bash (Linux/macOS):

```bash
docker run -p 6333:6333 -p 6334:6334 -v "$(pwd)/qdrant_storage:/qdrant/storage" qdrant/qdrant
```

In [5]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, SparseVectorParams, PointStruct, SparseVector

# Conectar con Qdrant
cliente = QdrantClient(host="localhost", port=6333)

collection_name = "rag_privacy_corpus"

# Recrear coleccion para evitar conflicto de esquema o ids previos
if cliente.collection_exists(collection_name=collection_name):
    cliente.delete_collection(collection_name=collection_name)

cliente.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": VectorParams(size=768, distance=Distance.COSINE)
    },
    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

print(f"Coleccion lista: {collection_name}")

Coleccion lista: rag_privacy_corpus


In [6]:
# Ingest: insertar los embeddings con metadata en Qdrant
puntos = []
for i, item in enumerate(embeddings_corpus):
    puntos.append(
        PointStruct(
            id=i,
            vector={
                "dense": item["embedding"].tolist(),
                "sparse": SparseVector(
                    indices=sparse_vectors[i].indices.tolist(),
                    values=sparse_vectors[i].values.tolist(),
                ),
            },
            payload={
                "chunk": item["chunk"],
                "archivo": item["archivo"],
                "categoria": item["categoria"],
                "empresa": item["empresa"],
            },
        )
    )

# Upsert: inserta o actualiza si el id ya existe
cliente.upsert(collection_name="rag_privacy_corpus", points=puntos)
print(f"Puntos insertados/actualizados: {len(puntos)}")

Puntos insertados/actualizados: 268


In [15]:
import numpy as np
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue
from sentence_transformers import CrossEncoder

# Inicializar el cross-encoder para reranking
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def buscar_qdrant(query, top_k_inicial=10, top_k_final=3, filtro_categoria=None, filtro_empresa=None, mostrar_chunks=False):
    """
    Realiza una búsqueda en Qdrant utilizando un pipeline completo de RAG:

    1. Búsqueda híbrida (dense + sparse/BM25):
       - Combina embeddings densos (E5) con representaciones sparse (BM25)
       - Los resultados de ambas búsquedas se fusionan mediante Reciprocal Rank Fusion (RRF)
       - Mejora la recuperación al capturar similitud semántica y coincidencias léxicas

    2. Filtrado por metadatos (opcional):
       - Permite restringir resultados por categoría o empresa

    3. Reranking con cross-encoder:
       - Reordena los resultados iniciales evaluando directamente la relación (query, chunk)
       - Mejora la precisión final del contexto

    Parameters:
        query (str): consulta del usuario
        top_k_inicial (int): número de candidatos recuperados inicialmente
        top_k_final (int): número de resultados tras reranking
        filtro_categoria (str, opcional): filtrar por categoría
        filtro_empresa (str, opcional): filtrar por empresa

    Returns:
        list[dict]: lista de chunks relevantes con metadata y scores
    """
    # --- 1. RETRIEVAL HÍBRIDO (dense + sparse)
    # Embedding denso (semántico)
    query_emb = modelo_emb.encode(["query: " + query], normalize_embeddings=True)[0]
    
    # Embedding disperso (coincidencia léxica)
    query_sparse_raw = list(sparse_model.embed([query]))[0]
    query_sparse = SparseVector(
        indices=query_sparse_raw.indices.tolist(),
        values=query_sparse_raw.values.tolist(),
    )

    # FILTRADO POR METADATA (opcional)
    filtro = None

    if filtro_categoria or filtro_empresa:
        must_conditions = []
        
        # Categorías (OR entre varias categorías)
        if filtro_categoria:
            if isinstance(filtro_categoria, list):
                # Si es una lista de categorías, se crea un bloque OR (should) para ellas
                categorias_cond = Filter(
                    should=[
                        FieldCondition(
                            key="categoria",
                            match=MatchValue(value=cat)
                        )
                        for cat in filtro_categoria
                    ]
                )
                must_conditions.append(categorias_cond)
            else:
                # Si es una única categoría, se añade directamente como condición
                must_conditions.append(
                    FieldCondition(
                        key="categoria",
                        match=MatchValue(value=filtro_categoria)
                    )
                )

        # Empresas (OR entre varias empresas)
        # Solo se aplica este filtro si la categoría es "terms" o "privacy_policies"
        if filtro_empresa and (filtro_categoria in ["terms", "privacy_policies"]):
            if isinstance(filtro_empresa, list):
                empresas_cond = Filter(
                    should=[
                        FieldCondition(
                            key="empresa",
                            match=MatchValue(value=emp)
                        )
                        for emp in filtro_empresa
                    ]
                )
                must_conditions.append(empresas_cond)
            else:
                must_conditions.append(
                    FieldCondition(
                        key="empresa",
                        match=MatchValue(value=filtro_empresa)
                    )
                )

        # Construcción final (AND entre bloques)
        filtro = Filter(must=must_conditions)

    # --- BÚSQUEDA EN QDRANT (HÍBRIDA)
    # Qdrant combina automáticamente dense + sparse en un único score híbrido
    resultados = cliente.query_points(
        collection_name="rag_privacy_corpus",
        prefetch=[
            # Recupera candidatos de forma independiente para cada tipo de representación
            models.Prefetch(
                query=query_emb.tolist(),
                using="dense",
                limit=top_k_inicial * 2, # Doble de candidatos para mejorar recall antes del reranking
            ),
            models.Prefetch(
                query=query_sparse,
                using="sparse",
                limit=top_k_inicial * 2,
            ),
        ],
        # Combina los rankings de dense y sparse usando RRF
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        # Se reduce el número de candidatos antes del reranking
        limit=top_k_inicial, 
        query_filter=filtro,
        with_payload=True,
    )

    # --- 2. RERANKING: el cross-encoder evalúa cada par (query, chunk) conjuntamente
    # Crear pares (query, chunk)
    pairs = [
        (query, punto.payload["chunk"])
        for punto in resultados.points
    ]

    # Obtener scores del reranker
    scores = reranker.predict(pairs)

    # Combinar resultados
    reranked = []
    for i, punto in enumerate(resultados.points):
        reranked.append({
            "chunk": punto.payload["chunk"],
            "categoria": punto.payload["categoria"],
            "archivo": punto.payload["archivo"],
            "empresa": punto.payload.get("empresa", None),
            # Score híbrido (dense + sparse combinado por Qdrant)
            "score_hybrid": punto.score,
            # Score del reranker (decisión final)
            "score_rerank": float(scores[i])
        })

    # Ordenar por score del reranker
    reranked = sorted(reranked, key=lambda x: x["score_rerank"], reverse=True)

    # Seleccionar top final
    reranked_final = reranked[:top_k_final]

    # --- DEBUG / PRINT
    if mostrar_chunks:
        print(f"Query: '{query}'")
        if filtro_categoria:
            print(f"Filtro: categoria = '{filtro_categoria}'")
        if filtro_empresa:
            print(f"Filtro: empresa = '{filtro_empresa}'")
        print()

        for i, r in enumerate(reranked_final):
            print(f"{i + 1}. (rerank={r['score_rerank']:.3f} | hybrid={r['score_hybrid']:.3f}) [{r['categoria']} - {r['archivo']}]")
            print(f"   {r['chunk']}\n")

    return reranked_final

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5518.67it/s]


In [6]:
# Búsqueda sin filtro
a = buscar_qdrant("¿Qué datos personales recopila Google?", mostrar_chunks=True)

Query: '¿Qué datos personales recopila Google?'

1. (rerank=0.725 | hybrid=0.200) [privacy_policies - google_privacy.md]
   When you’re signed in, we also collect information that we store with your Google Account, which we treat aspersonal information.

2. (rerank=0.294 | hybrid=0.312) [privacy_policies - spotify_privacy.md]
   | Datos de pago y compra | Si compras algo en Spotify o te registras en una Opción de Servicio de pago o a una prueba, necesitaremos procesar tus datos de pago. Los datos personales exactos recopilados y utilizados variarán en función de la forma de pago, pero incluirán información como la siguiente: Nombre Fecha de nacimiento Tipo de forma de pago (por ejemplo, tarjeta de crédito o débito) Si utilizas una tarjeta de débito o crédito, el tipo de tarjeta, la fecha de vencimiento y determinados dígitos del número de tu tarjeta. (Nota: Por motivos de seguridad, nunca almacenamos tu número de tarjeta completo). Código postal Número de teléfono móvil Información de 

In [ ]:
# Búsqueda con filtro por categoría
_ = buscar_qdrant("¿Qué datos personales recopila Google?", filtro_categoria="privacy_policies", mostrar_chunks=True)

Query: '¿Qué datos personales recopila Google?'
Filtro: categoria = 'privacy_policies'

1. (rerank=0.725 | hybrid=0.200) [privacy_policies - google_privacy.md]
   When you’re signed in, we also collect information that we store with your Google Account, which we treat aspersonal information.

2. (rerank=0.294 | hybrid=0.317) [privacy_policies - spotify_privacy.md]
   | Datos de pago y compra | Si compras algo en Spotify o te registras en una Opción de Servicio de pago o a una prueba, necesitaremos procesar tus datos de pago. Los datos personales exactos recopilados y utilizados variarán en función de la forma de pago, pero incluirán información como la siguiente: Nombre Fecha de nacimiento Tipo de forma de pago (por ejemplo, tarjeta de crédito o débito) Si utilizas una tarjeta de débito o crédito, el tipo de tarjeta, la fecha de vencimiento y determinados dígitos del número de tu tarjeta. (Nota: Por motivos de seguridad, nunca almacenamos tu número de tarjeta completo). Código postal N

In [ ]:
# Búsqueda con filtro por categoría + empresa
_ = buscar_qdrant("¿Qué datos personales recopila Google?", filtro_categoria="privacy_policies", filtro_empresa="Google", mostrar_chunks=True)
# Con ambos filtros, se consiguen resultados sean más precisos y relevantes

Query: '¿Qué datos personales recopila Google?'
Filtro: categoria = 'privacy_policies'
Filtro: empresa = 'Google'

1. (rerank=0.725 | hybrid=0.500) [privacy_policies - google_privacy.md]
   When you’re signed in, we also collect information that we store with your Google Account, which we treat aspersonal information.

2. (rerank=0.689 | hybrid=0.333) [privacy_policies - google_privacy.md]
   In some circumstances, Google also collects information about you frompublicly accessible sources. For example, if your name appears in your local newspaper, Google’s Search engine may index that article and display it to other people if they search for your name. We may also collect information about you from trusted partners, such as directory services who provide us with business information to be displayed on Google’s services, marketing partners who provide us with information about potential customers of our business services, and security partners who provide us with information toprotect a

In [29]:
# Pregunta fuera del dominio 
_ = buscar_qdrant("¿Cómo funciona la fotosíntesis en las plantas?")
# Devuelve chunks, pero el reranking deja el score a 0, mostrando que no son relevantes

Query: '¿Cómo funciona la fotosíntesis en las plantas?'

1. (rerank=0.000 | hybrid=0.250) [terms - spotify_terms.md]
   . POR OTRA PARTE, SPOTIFY NO HACE NINGUNA DECLARACIÓN, NI GARANTIZA NI ASUME RESPONSABILIDAD ALGUNA, EN RELACIÓN CON LAS APLICACIONES DE TERCEROS (O EL CONTENIDO DE LAS MISMAS), LOS CONTENIDOS DE LOS USUARIOS, LOS DISPOSITIVOS O CUALQUIER PRODUCTO O SERVICIO ANUNCIADO, PROMOCIONADO U OFRECIDO POR UN TERCERO EN EL SERVICIO O EN CUALQUIER SITIO WEB CON HIPERVÍNCULOS, O A TRAVÉS DE LOS MISMOS, Y SPOTIFY NO ES RESPONSABLE DE LAS TRANSACCIONES ENTRE USTED Y LOS PROVEEDORES EXTERNOS DE LO QUE ANTECEDE. NINGÚN CONSEJO NI INFORMACIÓN, VERBAL O POR ESCRITO, QUE USTED OBTENGA DE SPOTIFY DARÁ LUGAR A NINGUNA GARANTÍA EN NOMBRE DE SPOTIFY. DURANTE LA UTILIZACIÓN DEL SERVICIO DE SPOTIFY, ES POSIBLE QUE TENGA ACCESO A LAS FUNCIONES DE FILTRADO DE CONTENIDOS EXPLÍCITOS, PERO EL USO DE DICHAS FUNCIONES PODRÁ DAR LUGAR A QUE SE OFREZCA ALGÚN CONTENIDO EXPLÍCITO, POR LO QUE NO DEBERÁ C

### 3.3. Construcción del prompt para RAG

In [8]:
# Mismo prompt que se selecciomó en el apartado 1, pero adaptado para RAG
SYSTEM_PROMPT_RAG = (
    "Eres un asistente especializado en privacidad, protección de datos y términos de servicio."
    "Responde únicamente utilizando la información proporcionada en el contexto."
    "Si la información no es suficiente o la pregunta no está relacionada con privacidad, protección de datos o términos digitales, debes responder exactamente:"
    "\"No dispongo de información suficiente para responder con precisión.\""
    "No respondas preguntas fuera de este dominio bajo ninguna circunstancia."
    "Usa un tono formal y respuestas breves y estructuradas."
)

def construir_user_prompt(query, contexto):
    return f"""
Utiliza el siguiente contexto para responder a la pregunta del usuario.

CONTEXTO:
---
{contexto}
---

PREGUNTA:
{query}
"""

# --- FUNCIONES AUXILIARES PARA DETECCIÓN DE FILTROS
def detectar_empresas(query):
    empresas = []
    
    query_lower = query.lower()
    if "google" in query_lower:
        empresas.append("Google")
    if "spotify" in query_lower:
        empresas.append("Spotify")

    return empresas if empresas else None
    
# Descripciones para cada categoría
categoria_descripciones = {
    "guides": "explanations of privacy concepts like GDPR and cookies",
    "privacy_policies": "how companies collect and use personal data",
    "terms": "terms of service and usage conditions"
}

# Generar embeddings para cada descripción
categoria_embeddings = {}
for cat, desc in categoria_descripciones.items():
    emb = modelo_emb.encode(["passage: " + desc], normalize_embeddings=True)[0]
    categoria_embeddings[cat] = emb
    
def detectar_categorias(query, margin=0.02, min_score=0.20):
    query_emb = modelo_emb.encode(["query: " + query], normalize_embeddings=True)[0]

    # Calcular similitud coseno entre la query y cada descripción de categoría
    scores = {}
    for cat, desc_emb in categoria_embeddings.items():
        scores[cat] = np.dot(query_emb, desc_emb)

    # Umbral mínimo real: si no hay suficiente similitud, no aplicar filtro de categoría
    max_score = max(scores.values())
    if max_score < min_score:
        return None

    # Identificar categorías con score cercano al máximo (dentro de un margen)
    categorias = [
        cat for cat, score in scores.items()
        if (max_score - score) <= margin
    ]

    return categorias if categorias else None

### 3.4. Generar respuesta RAG

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
API_KEY = os.getenv("GROQ_KEY")

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=API_KEY)

MODEL = "llama-3.1-8b-instant"

def generar_respuesta_rag(pregunta):
    # Obtener contexto relevante de Qdrant
    resultados = buscar_qdrant(
        pregunta, 
        filtro_categoria=detectar_categorias(pregunta),
        filtro_empresa=detectar_empresas(pregunta), 
        mostrar_chunks=True
    )

    # Si todos los scores son bajos, nos ahorramos llamar al LLM
    if all(r["score_rerank"] <= 0.15 for r in resultados):
        return "Scores bajos: No dispongo de información suficiente para responder con precisión."

    # Construir contexto estructurado para el LLM, incluyendo metadatos
    contexto_list = []
    for i, r in enumerate(resultados):
        bloque = f"""[Fragmento {i+1} | {r['empresa']} | {r['categoria']} | {r['archivo']}]
{r['chunk']}"""
        contexto_list.append(bloque)

    contexto_str = "\n\n".join(contexto_list)
    
    # Generar respuesta con el LLM utilizando el contexto proporcionado
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_RAG},
            {"role": "user", "content": construir_user_prompt(pregunta, contexto_str)}
        ],
        max_completion_tokens=300,
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

### 3.5. Probar el funcionamiento

#### 3.5.1. Preguntas generales dentro del dominio

Estas preguntas permiten evaluar la capacidad del sistema RAG para recuperar chunks relevantes y responder con información precisa en casos donde la pregunta es general y sencilla, por lo que no obliga a aplicar filtros estrictos.

In [ ]:
respuesta = generar_respuesta_rag("¿Que son las cookies y para qué se utilizan?")
print("="*50)
print(respuesta)
# Responde correctamente utilizando principalmente el documento cookies.md, 
# también recupera un chunk de spotify_privacy.md que describe brevemente las cookies.

Query: '¿Que son las cookies y para qué se utilizan?'
Filtro: categoria = '['guides', 'privacy_policies']'

1. (rerank=0.961 | hybrid=1.000) [privacy_policies - spotify_privacy.md]
   - Política de cookies: información sobre cómo utilizamos las cookies y cómo gestionar tus preferencias de cookies. Las cookies son archivos que se guardan en tu teléfono, tablet u ordenador cuando visitas un sitio web.

2. (rerank=0.956 | hybrid=0.250) [guides - cookies_guide.md]
   ## What Are Internet Cookies?

Cookies (often known as internet cookies) are text files with small pieces of data — like a username and password — that are used to identify your computer as you use a network. Specific cookies are used to identify specific users and improve their web browsing experience. Data stored in a cookie is created by the server upon your connection. This data is labeled with an ID unique to you and your computer. When the cookie is exchanged between your computer and the network server, the server reads

In [ ]:
respuesta = generar_respuesta_rag("¿Qué derechos tienen los usuarios sobre sus datos personales?")
print("="*50)
print(respuesta)
# En este caso se espera que el modelo utilice principalmente el documento gdpr_explained.md,
# sin embargo, al no pedirselo explícitamente, usa solo chunks de spotify_privacy.md.
# Esto se debe seguramente a que el documento de Spotify es mucho mayor que el de GDPR, y el modelo tiende a priorizarlo.

Query: '¿Qué derechos tienen los usuarios sobre sus datos personales?'
Filtro: categoria = '['guides', 'privacy_policies']'

1. (rerank=0.959 | hybrid=0.833) [privacy_policies - spotify_privacy.md]
   ## 2. Derechos y controles sobre los datos personales

Muchas leyes de privacidad otorgan derechos a las personas sobre sus datos personales. Entre ellas se encuentra el Reglamento General de Protección de Datos, o "RGPD" .

Algunos derechos solo se aplican cuando Spotify utiliza una determinada "base jurídica" para tratar tus datos. Explicamos cada base jurídica y cuándo Spotify utiliza cada una de ellas en la sección 4 "Nuestra finalidad para el uso de tus datos personales" .

En la tabla siguiente se explican:

- Tus derechos
- Las circunstancias en las que se aplican (como la base jurídica necesaria)
- Cómo utilizarlos

2. (rerank=0.945 | hybrid=0.643) [privacy_policies - spotify_privacy.md]
   - Datos que conservamos hasta que los eliminas.
Tienes derecho a solicitar que eliminemos c

In [ ]:
respuesta = generar_respuesta_rag("¿Qué derechos tienen los usuarios sobre sus datos personales según el GDPR?")
print("="*50)
print(respuesta)
# Al incluir "según el GDPR" en la pregunta, se espera que el modelo priorice el documento gdpr_explained.md,
# y efectivamente lo hace, utilizando principalmente chunks de ese documento para responder con precisión a la pregunta. 
# Esto muestra la importancia de diseñar bien el prompt para guiar al modelo hacia la información más relevante.

Query: '¿Qué derechos tienen los usuarios sobre sus datos personales según el GDPR?'
Filtro: categoria = '['guides', 'privacy_policies']'

1. (rerank=0.966 | hybrid=0.250) [guides - gdpr_explained.md]
   ### People’s privacy rights

You are a data controller and/or a data processor. But as a person who uses the Internet, you’re also a data subject. The GDPR recognizes a litany of new privacy rights for data subjects , which aim to give individuals more control over the data they loan to organizations. As an organization, it’s important to understand these rights to ensure you are GDPR compliant.

Below is a rundown of data subjects’ privacy rights:

- The right to be informed
- The right of access
- The right to rectification
- The right to erasure
- The right to restrict processing
- The right to data portability
- The right to object
- Rights in relation to automated decision making and profiling.

2. (rerank=0.965 | hybrid=0.333) [guides - gdpr_explained.md]
   ---

[IMAGE: GPDR_rig

In [22]:
respuesta = generar_respuesta_rag("¿Qué datos recopilan las empresas cuando usamos sus servicios?")
print("="*50)
print(respuesta)
# En este caso, se espera que el modelo consulte documentos de Google y Spotify sobre privacy_policies.
# Sin embargo, el modelo se centra principalmente en el documento de Spotify.

Query: '¿Qué datos recopilan las empresas cuando usamos sus servicios?'
Filtro: categoria = '['privacy_policies']'

1. (rerank=0.457 | hybrid=0.833) [privacy_policies - spotify_privacy.md]
   | Colaboradores de publicidad y marketing | Recibimos suposiciones acerca de tus intereses y preferencias por parte de determinados colaboradores de publicidad o marketing. Esa información nos permite ofrecerte anuncios y contenido de marketing más relevantes. | Datos de uso |
| Empresas adquiridas | Es posible que recibamos datos sobre ti de las empresas que adquiramos con el objetivo de mejorar nuestros servicios, productos y ofertas. | Datos de usuario Datos de uso |

2. (rerank=0.391 | hybrid=0.417) [privacy_policies - spotify_privacy.md]
   | Elaborar planes, informes y previsiones de la empresa Por ejemplo, cuando analizamos datos agregados de usuarios, como el número de nuevos registros en un país, al planificar nuevas ubicaciones en las que lanzar nuestros productos y funciones. | Interés 

In [ ]:
respuesta = generar_respuesta_rag("What data do companies collect when we use their services?")
print("="*50)
print(respuesta)
# Al hacer la misma pregunta en inglés, se observa que el modelo recupera solo chunks del documento de Google, que está en inglés, y no del de Spotify, que está en español.
# Conclusion: el idioma de la pregunta influye en la selección de los documentos relevantes.

Query: 'What data do companies collect when we use their services?'
Filtro: categoria = '['privacy_policies']'

1. (rerank=0.773 | hybrid=0.233) [privacy_policies - google_privacy.md]
   ### Your location information

We collect location information when you use our services, which helps us offer features like driving directions, search results for things near you, and ads based on your location.

Depending on the products you’re using and settings you choose, Google may use different types of location information to help make some services and products you use more helpful. These include:

2. (rerank=0.732 | hybrid=0.202) [privacy_policies - google_privacy.md]
   ## Information we collect as you use our services
### Your apps, browsers & devices

We collect information about the apps, browsers, anddevicesyou use to access Google services, which helps us provide features like automatic product updates and dimming your screen if your battery runs low.

The information we collect include

#### 3.5.2. Preguntas específicas

Con estas preguntas se busca ver que el RAG sea capaz de aplicar filtros por categoría y empresa para recuperar información más precisa, y que el modelo utilice correctamente esa información para responder.

In [ ]:
respuesta = generar_respuesta_rag("¿Qué datos recopila Google cuando usamos sus servicios?")
print("="*50)
print(respuesta)
# Con la pregunta general en español, el modelo se centraba principalmente en el documento de Spotify.
# Al incluir "Google" en la pregunta, el RAG filtra por la empresa Google y devuelve información correcta.

Query: '¿Qué datos recopilan Google cuando usamos sus servicios?'
Filtro: categoria = '['guides', 'privacy_policies']'
Filtro: empresa = '['Google']'

1. (rerank=0.913 | hybrid=0.167) [privacy_policies - google_privacy.md]
   # Information Google collects
## We want you to understand the types of information we collect as you use our services

We collect information to provide better services to all our users — from figuring out basic stuff like which language you speak, to more complex things like whichads you’ll find most useful,the people who matter most to you online, or which YouTube videos you might like. The information Google collects, and how that information is used, depends on how you use our services and how you manage your privacy controls.

When you’re not signed in to a Google Account, we store the information we collect withunique identifierstied to the browser, application, ordeviceyou’re using. This allows us to do things like maintain your preferences across browsing

In [ ]:
respuesta = generar_respuesta_rag("¿Qué requisitos hay que cumplir para usar Spotify?")
print("="*50)
print(respuesta)
# Filtra correctamente por categoría "terms" y empresa "Spotify", y responde con precisión.

Query: '¿Qué requisitos hay que cumplir para usar Spotify?'
Filtro: categoria = '['terms']'
Filtro: empresa = '['Spotify']'

1. (rerank=0.999 | hybrid=0.667) [terms - spotify_terms.md]
   Para poder utilizar el Servicio de Spotify y acceder a cualquier Contenido, deberá (1) tener 13 años de edad (o la edad mínima equivalente en su país de origen) o más, (2) contar con el consentimiento de sus padres o de su tutor legal si es menor de edad en su país de origen; (3) estar facultado para celebrar un contrato vinculante con nosotros y no estar impedido para ello en virtud de la legislación aplicable, y (4) residir en un país en el que el Servicio esté disponible. También se compromete a que cualquier información de registro que envíe a Spotify es, y seguirá siendo en todo momento, verdadera, precisa y completa. Si es menor de edad en su país de origen, su padre, madre o tutor deberá aceptar estos Términos en su nombre. Si no cumple los requisitos de edad mínima para utilizar el Servicio de

In [11]:
respuesta = generar_respuesta_rag("¿Puede Google bloquear una cuenta de usuario?")
print("="*50)
print(respuesta)

Query: '¿Puede Google bloquear una cuenta de usuario?'
Filtro: categoria = '['guides', 'terms']'
Filtro: empresa = '['Google']'

1. (rerank=0.343 | hybrid=0.625) [terms - google_terms.md]
   ### Suspending or terminating your access to Google services

Without limiting any of our other rights, Google may suspend or terminate your access to the services or delete your Google Account if any of these things happen:

- you materially or repeatedly breach these terms, any additional terms specific to the services or policies.
- we’re required to do so to comply with a legal requirement or a court order
- we reasonably believe that your conduct causes harm or
For more information about why we disable accounts and what happens when we do, see this Help Center page . If you believe your Google Account has been suspended or terminated in error, you can appeal your decision.

Of course, you’re always free to stop using our services at any time. If you’re an EEA-based consumer , you can also with

In [ ]:
respuesta = generar_respuesta_rag("¿Qué derechos tienen los usuarios sobre sus datos según el GDPR y cómo pueden ejercerlos en Spotify?")
print("="*50)
print(respuesta)
# Se espera que el modelo utilice información de ambos documentos, gdpr_explained.md y spotify_privacy.md, 
# pero solo usa información de spotify_privacy.md. 
# Aun así la respuesta que da es correcta, aunque alucina al decir "Según el GDPR", ya que no está utilizando información directa de gdpr_explained.md.

Query: '¿Qué derechos tienen los usuarios sobre sus datos según el GDPR y cómo pueden ejercerlos en Spotify?'
Filtro: categoria = '['guides', 'privacy_policies', 'terms']'
Filtro: empresa = '['Spotify']'

1. (rerank=0.919 | hybrid=1.000) [privacy_policies - spotify_privacy.md]
   ## 2. Derechos y controles sobre los datos personales

Muchas leyes de privacidad otorgan derechos a las personas sobre sus datos personales. Entre ellas se encuentra el Reglamento General de Protección de Datos, o "RGPD" .

Algunos derechos solo se aplican cuando Spotify utiliza una determinada "base jurídica" para tratar tus datos. Explicamos cada base jurídica y cuándo Spotify utiliza cada una de ellas en la sección 4 "Nuestra finalidad para el uso de tus datos personales" .

En la tabla siguiente se explican:

- Tus derechos
- Las circunstancias en las que se aplican (como la base jurídica necesaria)
- Cómo utilizarlos

2. (rerank=0.899 | hybrid=0.444) [privacy_policies - spotify_privacy.md]
   - Datos que

In [ ]:
# Pregunta doble que requiere información de ambos documentos
respuesta = generar_respuesta_rag("¿Qué edad mínima se requiere para usar Spotify? ¿Y para usar Google?")
print("="*50)
print(respuesta)
# Solo recupera chunks de Spotify (aunque sí hay información útil de Google en google_terms.md), 
# por lo que solo responde a la edad mínima para usar Spotify, e indica que no tiene información para Google.

Query: '¿Qué edad mínima se requiere para usar Spotify? ¿Y para usar Google?'
Filtro: categoria = '['guides', 'terms']'
Filtro: empresa = '['Google', 'Spotify']'

1. (rerank=0.991 | hybrid=1.000) [terms - spotify_terms.md]
   Para poder utilizar el Servicio de Spotify y acceder a cualquier Contenido, deberá (1) tener 13 años de edad (o la edad mínima equivalente en su país de origen) o más, (2) contar con el consentimiento de sus padres o de su tutor legal si es menor de edad en su país de origen; (3) estar facultado para celebrar un contrato vinculante con nosotros y no estar impedido para ello en virtud de la legislación aplicable, y (4) residir en un país en el que el Servicio esté disponible. También se compromete a que cualquier información de registro que envíe a Spotify es, y seguirá siendo en todo momento, verdadera, precisa y completa. Si es menor de edad en su país de origen, su padre, madre o tutor deberá aceptar estos Términos en su nombre. Si no cumple los requisitos de ed

In [ ]:
# Misma pregunta pero cambiando el orden de las empresas.
respuesta = generar_respuesta_rag("¿Qué edad mínima se requiere para usar Google? ¿Y para usar Spotify?")
print("="*50)
print(respuesta)
# Sigue recuperando solo chunks de Spotify y sin información de Google.

Query: '¿Qué edad mínima se requiere para usar Google? ¿Y para usar Spotify?'
Filtro: categoria = '['guides', 'terms']'
Filtro: empresa = '['Google', 'Spotify']'

1. (rerank=0.988 | hybrid=1.000) [terms - spotify_terms.md]
   Para poder utilizar el Servicio de Spotify y acceder a cualquier Contenido, deberá (1) tener 13 años de edad (o la edad mínima equivalente en su país de origen) o más, (2) contar con el consentimiento de sus padres o de su tutor legal si es menor de edad en su país de origen; (3) estar facultado para celebrar un contrato vinculante con nosotros y no estar impedido para ello en virtud de la legislación aplicable, y (4) residir en un país en el que el Servicio esté disponible. También se compromete a que cualquier información de registro que envíe a Spotify es, y seguirá siendo en todo momento, verdadera, precisa y completa. Si es menor de edad en su país de origen, su padre, madre o tutor deberá aceptar estos Términos en su nombre. Si no cumple los requisitos de ed

In [ ]:
# Misma pregunta en inglés
respuesta = generar_respuesta_rag("What is the minimum age required to use Google? And for Spotify?")
print("="*50)
print(respuesta)
# Al hacer la pregunta en inglés, si recupera chunks de Google y responde correctamente a ambas partes de la pregunta.
# Esto refuerza la conclusión de que el idioma de la pregunta influye en la selección de los documentos relevantes, y por tanto en la respuesta final del modelo.
# Además, aunque se le pregunta en inglés, el modelo responde en español, probablemente debido a que el system prompt está en español.

Query: 'What is the minimum age required to use Google? And for Spotify?'
Filtro: categoria = '['guides', 'terms']'
Filtro: empresa = '['Google', 'Spotify']'

1. (rerank=0.961 | hybrid=0.333) [terms - spotify_terms.md]
   Para poder utilizar el Servicio de Spotify y acceder a cualquier Contenido, deberá (1) tener 13 años de edad (o la edad mínima equivalente en su país de origen) o más, (2) contar con el consentimiento de sus padres o de su tutor legal si es menor de edad en su país de origen; (3) estar facultado para celebrar un contrato vinculante con nosotros y no estar impedido para ello en virtud de la legislación aplicable, y (4) residir en un país en el que el Servicio esté disponible. También se compromete a que cualquier información de registro que envíe a Spotify es, y seguirá siendo en todo momento, verdadera, precisa y completa. Si es menor de edad en su país de origen, su padre, madre o tutor deberá aceptar estos Términos en su nombre. Si no cumple los requisitos de edad m

In [ ]:
# Pregunta comparativa que requiere información de ambos documentos
respuesta = generar_respuesta_rag("Which is more privacy-friendly: Google or Spotify?")
print("="*50)
print(respuesta)
# El RAG falla al no recuperar ningún chunk relevante

Query: 'Which is more privacy-friendly: Google or Spotify?'
Filtro: categoria = '['guides', 'privacy_policies']'
Filtro: empresa = '['Google', 'Spotify']'

1. (rerank=0.024 | hybrid=0.833) [privacy_policies - google_privacy.md]
   - Google apps, sites, and devices, like Search, YouTube, and Google Home
- Platforms like the Chrome browser and Android operating system
- Products that are integrated into third-party apps and sites, like ads, analytics, and embedded Google Maps
You can use our services in a variety of ways to manage your privacy. For example, you can sign up for a Google Account if you want to create and manage content like emails and photos, or see more relevant search results. And you can use many Google services when you’re signed out or without creating an account at all, like searching on Google or watching YouTube videos. You can also choose to browse the web in a private mode, likeChrome Incognitomode, which helps keep your browsing private from other people who use

#### 3.5.3. Preguntas sobre información muy concreta

Con estas preguntas se busca evaluar la capacidad del RAG para recuperar información muy específica que aparece en una sola línea o pocas líneas de un documento concreto.

In [ ]:
# Aparece en una pequeña sección al final de spotify_privacy.md 
respuesta = generar_respuesta_rag("¿Cómo puedo ponerme en contacto con Spotify si tengo dudas sobre su política de privacidad?")
print("="*50)
print(respuesta)
# El RAG recupera el chunk específico de spotify_privacy.md y responde correctamente a la pregunta utilizando esa información.

Query: '¿Cómo puedo ponerme en contacto con Spotify si tengo dudas sobre su política de privacidad?'
Filtro: categoria = '['guides', 'privacy_policies', 'terms']'
Filtro: empresa = '['Spotify']'

1. (rerank=0.999 | hybrid=1.000) [privacy_policies - spotify_privacy.md]
   ## 11. Cómo ponerte en contacto con nosotros

Si tienes preguntas o dudas sobre esta Política, ponte en contacto con nuestro responsable de protección de datos de cualquiera de las siguientes formas:

- Envía un correo electrónico a privacy@spotify.com.
- Escríbenos a Spotify AB, Regeringsgatan 19, 111 53 Estocolmo, Suecia.
Spotify AB es el responsable del tratamiento de los datos personales tratados en virtud de esta Política.

2. (rerank=0.861 | hybrid=0.300) [privacy_policies - spotify_privacy.md]
   | Retirada del consentimiento | Retirarnos el consentimiento para que recopilemos o utilicemos tus datos personales. Puedes hacerlo si Spotify trata tus datos personales según la base jurídica del consentimiento. | Para

In [ ]:
# Pregunta muy concreta cuya información aparece solo en una linea
respuesta = generar_respuesta_rag("¿Cómo puedo ponerme en contacto con Google desde la Polinesia Francesa para ejercer mi derecho de desistimiento?")
print("="*50)
print(respuesta)
# El RAG recupera correctamente el chunk relevante y responde correctamente a la pregunta.

Query: '¿Cómo puedo ponerme en contacto con Google desde la Polinesia Francesa para ejercer mi derecho de desistimiento?'
Filtro: categoria = '['guides', 'privacy_policies', 'terms']'
Filtro: empresa = '['Google']'

1. (rerank=0.549 | hybrid=0.500) [terms - google_terms.md]
   ### Contact Google to withdraw from these terms
| Country | Phone number |
| --- | --- |
| Austria | 0800 001180 |
| Åland Islands | 0800 526683 |
| Belgium | 0800 58 142 |
| Bulgaria | 0800 14 744 |
| Canary Islands | +34 912 15 86 27 |
| Ceuta & Melilla | +34 912 15 86 27 |
| Croatia | 0800 787 086 |
| Cyprus | 80 092492 |
| Czechia | 800 720 070 |
| Denmark | 80 40 01 11 |
| Estonia | 8002 643 |
| Finland | 0800 520030 |
| France | 0 805 98 03 38 |
| French Guiana | 0805 98 03 38 |
| French Polynesia | +33 1 85 14 96 65 |
| French Southern Territories | +33 1 85 14 96 65 |
| Germany | 0800 6270502 |
| Greece | 21 1180 9433 |
| Guadeloupe | 0805 98 03 38 |
| Hungary | 06 80 200 148 |
| Iceland | 800 4177 |
| Ir

In [ ]:
# Pregunta doble sobre dos informaciones concretas que aparacen en una linea cada documento
respuesta = generar_respuesta_rag("¿Desde cuando está en vigor la política de privacidad de Spotify? ¿Y sus términos de servicio?")
print("="*50)
print(respuesta)
# Solo recupera el chunk sobre los terminos de servcio, pero no el de la política de privacidad, por lo que solo responde a la segunda parte de la pregunta.
# Esto se debe a que ha dejado "privacy_policies" fuera de los filtros, por lo que no se está considerando ese documento al generar la respuesta.

Query: '¿Desde cuando está en vigor la política de privacidad de Spotify? ¿Y sus términos de servicio?'
Filtro: categoria = '['guides', 'terms']'
Filtro: empresa = '['Spotify']'

1. (rerank=0.865 | hybrid=0.556) [terms - spotify_terms.md]
   # Términos y Condiciones de uso de Spotify

Última actualización: 26/08/2025
## 1. Introducción

Lea detenidamente los presentes Términos y Condiciones de uso (en lo sucesivo, los << Términos >>) que rigen el uso que usted haga de los servicios personalizados de Spotify (que incluyen el acceso a los mismos) para la transmisión de música y otros contenidos, incluidos todos nuestros sitios web y aplicaciones de software que incorporan o enlazan con los presentes Términos (en conjunto, el << Servicio de Spotify >>) y cualquier música, vídeo, podcast, audiolibro u otro material disponible a través del Servicio de Spotify (en lo sucesivo, el << Contenido >>).

2. (rerank=0.846 | hybrid=0.833) [terms - spotify_terms.md]
   El uso del Servicio de Spotify 

In [23]:
# Misma pregunta pero sin la segunda parte, para ver si así recupera el chunk relevante de la política de privacidad.
respuesta = generar_respuesta_rag("¿Desde cuando está en vigor la política de privacidad de Spotify?")
print("="*50)
print(respuesta)
# Al eliminar la segunda parte de la pregunta, el RAG recupera el chunk relevante de spotify_privacy.md y responde correctamente a la pregunta.

Query: '¿Desde cuando está en vigor la política de privacidad de Spotify?'
Filtro: categoria = '['guides', 'privacy_policies', 'terms']'
Filtro: empresa = '['Spotify']'

1. (rerank=0.998 | hybrid=1.000) [privacy_policies - spotify_privacy.md]
   # Política de privacidad de Spotify

En vigor desde el 27 de agosto de 2025
## 1. Acerca de esta Política

En esta Política de privacidad, se describe cómo Spotify AB trata tus datos personales. De ahora en adelante, nos referiremos a ella como la "Política".

Se aplica al uso por tu parte de lo siguiente:

- Todos los servicios de streaming de Spotify como usuario. Entre ellos, se incluyen:
  - El uso de Spotify por tu parte en cualquier dispositivo.
  - La personalización de tu experiencia de usuario.
  - La infraestructura necesaria para proporcionar nuestros servicios.
  - La conexión de tu cuenta de Spotify con otra aplicación.
  - Nuestras opciones de streaming gratuitas y de pago (cada una de ellas una " Opción de servicio ").
- Otros se

#### 3.5.4. Preguntas fuera de dominio

In [ ]:
respuesta = generar_respuesta_rag("¿Cómo funciona la fotosíntesis de las plantas?")
print("="*50)
print(respuesta)
# En este caso, los chunks recuperados tiene un score muy bajo, por lo que no se envía la pregunta al LLM 
# y directamente se indica que no se ha encontrado información relevante.

Query: '¿Cómo funciona la fotosíntesis de las plantas?'
Filtro: categoria = '['guides', 'privacy_policies', 'terms']'

1. (rerank=0.000 | hybrid=0.250) [terms - spotify_terms.md]
   . POR OTRA PARTE, SPOTIFY NO HACE NINGUNA DECLARACIÓN, NI GARANTIZA NI ASUME RESPONSABILIDAD ALGUNA, EN RELACIÓN CON LAS APLICACIONES DE TERCEROS (O EL CONTENIDO DE LAS MISMAS), LOS CONTENIDOS DE LOS USUARIOS, LOS DISPOSITIVOS O CUALQUIER PRODUCTO O SERVICIO ANUNCIADO, PROMOCIONADO U OFRECIDO POR UN TERCERO EN EL SERVICIO O EN CUALQUIER SITIO WEB CON HIPERVÍNCULOS, O A TRAVÉS DE LOS MISMOS, Y SPOTIFY NO ES RESPONSABLE DE LAS TRANSACCIONES ENTRE USTED Y LOS PROVEEDORES EXTERNOS DE LO QUE ANTECEDE. NINGÚN CONSEJO NI INFORMACIÓN, VERBAL O POR ESCRITO, QUE USTED OBTENGA DE SPOTIFY DARÁ LUGAR A NINGUNA GARANTÍA EN NOMBRE DE SPOTIFY. DURANTE LA UTILIZACIÓN DEL SERVICIO DE SPOTIFY, ES POSIBLE QUE TENGA ACCESO A LAS FUNCIONES DE FILTRADO DE CONTENIDOS EXPLÍCITOS, PERO EL USO DE DICHAS FUNCIONES PODRÁ DAR LUGAR A QU

In [ ]:
respuesta = generar_respuesta_rag("¿Cuál es la capital de Francia?")
print("="*50)
print(respuesta)
# Mismo resultado que la pregunta anterior

Query: '¿Cuál es la capital de Francia?'
Filtro: categoria = '['guides', 'terms']'

1. (rerank=0.004 | hybrid=0.500) [terms - google_terms.md]
   | Martinique | 0805 98 03 38 |
| Mayotte | +33 1 85 14 96 65 |
| Netherlands | 0800 3600010 |
| New Caledonia | +33 1 85 14 96 65 |
| Norway | 800 62 068 |
| Poland | 800 410 575 |
| Portugal | 808 203 430 |
| Réunion | 0805 98 03 38 |
| Romania | 0800 672 350 |
| Slovakia | 0800 500 932 |
| Slovenia | 080 688882 |
| Spain | 900 906 451 |
| St. Barthélemy | +33 1 85 14 96 65 |
| St. Martin | +33 1 85 14 96 65 |
| St. Pierre & Miquelon | +33 1 85 14 96 65 |
| Svalbard & Jan Mayen | 800 62 425 |
| Sweden | 020-012 52 41 |
| Vatican City | 800 599 102 |
| Wallis & Futuna | +33 1 85 14 96 65 |

2. (rerank=0.000 | hybrid=0.500) [terms - spotify_terms.md]
   ### Usted es el único responsable de todo el Contenido de usuario que publique

Promete que, con respecto a cualquier Contenido de usuario que publique en Spotify, (1) es propietario de dicho C

#### 3.5.5. Preguntas relacionadas con el dominio pero que no estan en el corpus

Con estas preguntas se intenta forzar que el modelo alucine una respuesta.

In [ ]:
respuesta = generar_respuesta_rag("¿Qué datos recopila Netflix cuando usamos sus servicios?")
print("="*50)
print(respuesta)
# En lugar de alucinar información incorrecta sobre Netflix, el modelo indica que no dispone de información 
# suficiente para responder con precisión, ya que no se han recuperado chunks relevantes.

Query: '¿Qué datos recopila Netflix cuando usamos sus servicios?'
Filtro: categoria = '['guides', 'privacy_policies']'

1. (rerank=0.060 | hybrid=0.583) [privacy_policies - spotify_privacy.md]
   | Datos de uso | Datos personales sobre ti recopilados y tratados cuando accedes al Servicio Spotify o lo usas. Se incluyen los siguientes tipos de información: Información sobre cómo utilizas Spotify Estos son algunos ejemplos: Información sobre tu Opción de servicio de Spotify Acciones que realizas con el Servicio de Spotify (incluidas la fecha y la hora) como, por ejemplo, las siguientes: Búsquedas Historial de streaming Listas de reproducción que creas Tu biblioteca Historial de navegación Configuración de la cuenta Interacciones con otros usuarios de Spotify El uso por tu parte de servicios, dispositivos y aplicaciones de terceros en relación con el Servicio de Spotify Inferencias deductivas (es decir, lo que interpretamos) sobre tu edad, tus intereses y tus preferencias en función del us

In [ ]:
respuesta = generar_respuesta_rag("¿Cómo puedo crear una cuenta en Spotify?")
print("="*50)
print(respuesta)
# En ningún documento del corpus se explica esto explícitamente. 
# Aun así el RAG recupera chunks que mencionan los datos que recopila Spotify y responde a partir de esa información.

Query: '¿Cómo puedo crear una cuenta en Spotify?'
Filtro: categoria = '['guides', 'privacy_policies', 'terms']'
Filtro: empresa = '['Spotify']'

1. (rerank=0.953 | hybrid=0.559) [terms - spotify_terms.md]
   ## 3. Uso del servicio de Spotify

Para utilizar la totalidad o parte del Servicio de Spotify, es posible que tenga que crear una cuenta de Spotify, y dicha cuenta debe corresponder a su país de residencia. Su nombre de usuario y su contraseña son exclusivamente para su uso personal y deben ser confidenciales. Usted reconoce que es responsable de todo uso de su nombre de usuario y su contraseña, incluyendo cualquier uso no autorizado. Notifique inmediatamente a nuestro equipo de Atención al cliente si pierde o le roban su nombre de usuario o contraseña, o si cree que se ha producido un acceso no autorizado a su cuenta.

Spotify puede exigirle que cambie su nombre de usuario por cualquier motivo.

2. (rerank=0.895 | hybrid=0.500) [privacy_policies - spotify_privacy.md]
   | Datos de

In [ ]:
# Pregunta absurda que mezcla información real con un elemento absurdo (gato), para ver si el modelo alucina.
respuesta = generar_respuesta_rag("¿Qué requisitos tiene que cumplir un gato para usar Spotify?")
print("="*50)
print(respuesta)
# El modelo recupera información sobre los requisitos para usar Spotify, pero indica que no dispone de información suficiente para responder.

Query: '¿Qué requisitos tiene que cumplir un gato para usar Spotify?'
Filtro: categoria = '['guides', 'terms']'
Filtro: empresa = '['Spotify']'

1. (rerank=0.666 | hybrid=0.833) [terms - spotify_terms.md]
   Para poder utilizar el Servicio de Spotify y acceder a cualquier Contenido, deberá (1) tener 13 años de edad (o la edad mínima equivalente en su país de origen) o más, (2) contar con el consentimiento de sus padres o de su tutor legal si es menor de edad en su país de origen; (3) estar facultado para celebrar un contrato vinculante con nosotros y no estar impedido para ello en virtud de la legislación aplicable, y (4) residir en un país en el que el Servicio esté disponible. También se compromete a que cualquier información de registro que envíe a Spotify es, y seguirá siendo en todo momento, verdadera, precisa y completa. Si es menor de edad en su país de origen, su padre, madre o tutor deberá aceptar estos Términos en su nombre. Si no cumple los requisitos de edad mínima para uti

In [ ]:
respuesta = generar_respuesta_rag("¿Qué métodos de encriptado utiliza Google para proteger los datos de los usuarios?")
print("="*50)
print(respuesta)
# Responde que Google utiliza encriptación, pero no que métodos específicos, lo cual es correcto ya que no se proporciona esa información en el corpus.

Query: '¿Qué métodos de encriptado utiliza Google para proteger los datos de los usuarios?'
Filtro: categoria = '['guides', 'privacy_policies']'
Filtro: empresa = '['Google']'

1. (rerank=0.742 | hybrid=0.333) [privacy_policies - google_privacy.md]
   We work hard to protect you and Google from unauthorized access, alteration, disclosure, or destruction of information we hold, including:

- We use encryption to keep your data private while in transit
- We offer a range of security features, likeSafe Browsing, Security Checkup, and2 Step Verificationto help you protect your account
- We review our information collection, storage, and processing practices, including physical security measures, to prevent unauthorized access to our systems
- We restrict access to personal information to Google employees, contractors, and agents who need that information in order to process it. Anyone with this access is subject to strict contractual confidentiality obligations and may be disciplined or te

In [ ]:
# Pregunta que incluye un supuesto falso para intentar inducir al modelo a alucinar información incorrecta.
respuesta = generar_respuesta_rag("¿Por qué Google publica datos de usuarios sin su consentimiento?")
print("="*50)
print(respuesta)

Query: '¿Por qué Google publica datos de usuarios sin su consentimiento?'
Filtro: categoria = '['guides', 'privacy_policies']'
Filtro: empresa = '['Google']'

1. (rerank=0.361 | hybrid=0.333) [privacy_policies - google_privacy.md]
   ## When Google shares your information

We do not share your personal information with companies, organizations, or individuals outside of Google except in the following cases:

### With your consent

We’ll share personal information outside of Google when we have your consent. For example, if youuse Google Home to make a reservationthrough a booking service, we’ll get your permission before sharing your name or phone number with the restaurant. We also provide you with controls toreview and manage third party apps and sitesyou have given access to data in your Google Account. We’ll ask for your explicit consent to share anysensitive personal information.

2. (rerank=0.255 | hybrid=1.000) [privacy_policies - spotify_privacy.md]
   | Para fines de marketing

In [49]:
# Pregunta abierta que invita al modelo a especular y alucinar información
respuesta = generar_respuesta_rag("How do companies typically exploit user data for profit?")
print("="*50)
print(respuesta)

Query: 'How do companies typically exploit user data for profit?'
Filtro: categoria = '['privacy_policies']'

1. (rerank=0.003 | hybrid=0.433) [privacy_policies - google_privacy.md]
   ### Protect Google, our users, and the public

We use information to help improve thesafety and reliabilityof our services. This includes detecting, preventing, and responding to fraud, abuse, security risks, and technical issues that could harm Google, our users, orthe public.

We use different technologies to process your information for these purposes. We use automated systems that analyze your content to provide you with things like customized search results, personalized ads, or other features tailored to how you use our services. And we analyze your content to help usdetect abusesuch as spam, malware, and illegal content. We also usealgorithmsto recognize patterns in data. For example, Google Translate helps people communicate across languages by detecting common language patterns in phrases you as

In [ ]:
# Pregunta vaga para ver si el modelo alucina información concreta o responde de forma general indicando que no tiene suficiente información.
respuesta = generar_respuesta_rag("What happens with my data?")
print("="*50)
print(respuesta)
# Responde con información verdadera sin alucinar, pero solo sobre Google

Query: 'What happens with my data?'
Filtro: categoria = '['guides', 'privacy_policies']'

1. (rerank=0.748 | hybrid=0.183) [privacy_policies - google_privacy.md]
   - Some data you can delete whenever you like, such as yourpersonal infoor the content you create or upload, likephotosanddocuments. You can also deleteactivity informationsaved in your account, orchoose to have it deleted automaticallyafter a set period of time. We’ll keep this data in your Google Account until you remove it or choose to have it removed.
- Other data is deleted or anonymized automatically after a set period of time, such asadvertising datain server logs.
- We keep some data until you delete your Google Account, such as information about how often you use our services.
- And some data we retain for longer periods of time when necessary for legitimate business or legal purposes, such as security, fraud and abuse prevention, or financial record-keeping.

2. (rerank=0.325 | hybrid=0.500) [privacy_policies - goo

In [ ]:
# Pregunta con contradicción (cookies are not used for tracking) para ver si el modelo alucina una respuesta que intente resolver la contradicción.
respuesta = generar_respuesta_rag("If cookies are not used for tracking, why do websites use them?")
print("="*50)
print(respuesta)
# Responde ignorando la contradicción

Query: 'If cookies are not used for tracking, why do websites use them?'
Filtro: categoria = '['guides', 'privacy_policies']'

1. (rerank=0.841 | hybrid=0.750) [guides - cookies_guide.md]
   To be more concise, cookies are intended to be used for:

1. Session management: For example, cookies let websites recognize users and recall their individual login information and preferences, such as sports news versus politics.
2. Personalization: Customized advertising is the main way cookies are used to personalize your sessions. You may view certain items or parts of a site, and cookies use this data to help build targeted ads that you might enjoy. They’re also used for language preferences as well.
3. Tracking: Shopping sites use cookies to track items users previously viewed, allowing the sites to suggest other goods they might like and keep items in shopping carts while they continue shopping on another part of the website. They will also track and monitor performance analytics, like how m

In [ ]:
# Pedir al modelo que complete una frase que implica un supuesto falso.
respuesta = generar_respuesta_rag("Complete the sentence: 'Spotify sells all user data to third parties, so...'")
print("="*50)
print(respuesta)

Query: 'Complete the sentence: 'Spotify sells all user data to third parties, so...''
Filtro: categoria = '['guides', 'privacy_policies', 'terms']'
Filtro: empresa = '['Spotify']'

1. (rerank=0.004 | hybrid=0.167) [guides - gdpr_explained.md]
   Data subject — The person whose data is processed. These are your customers or site visitors.

Data controller — The person who decides why and how personal data will be processed. If you’re an owner or employee in your organization who handles data, this is you.

Data processor — A third party that processes personal data on behalf of a data controller. The GDPR has special rules for these individuals and organizations. These could include cloud servers, like Google Drive , Proton Drive , or Microsoft OneDrive, or email service providers, like Proton Mail .

2. (rerank=0.002 | hybrid=0.200) [privacy_policies - google_privacy.md]
   . Or we may let you know about upcoming changes or improvements to our services. And if you contact Google, we’ll